# Problem 2 — Transformer from scratch

All attention, multi-head attention, and encoder-block code is written from tensor operations.
`nn.MultiheadAttention` and `nn.Transformer*` are NOT used (only to verify).

Sections:
- P2.1 Scaled dot-product attention + unit tests + 3-token hand example
- P2.2 Multi-head attention + pre-LN encoder block + sinusoidal PE + parameter count
- P2.3 Task A: encoder classifier on HAR (positional ablation: none / sinusoidal / learned)
- P2.4 Task B: character-level causal LM on Tiny Shakespeare
- P2.5 Attention heatmaps, heads ablation, no-mask study, context-length time/memory


In [ ]:
import os, random, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = [0, 1, 2]
BATCH = 64
LR = 3e-4
FAST = False

print('Device:', DEVICE, '| CUDA:', torch.cuda.is_available())
print('PyTorch:', torch.__version__)


In [ ]:
def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

RESULTS_CSV = 'results.csv'

def log_run(**kw):
    cols = ['problem','model','seed','seq_len','test_acc','macro_f1','test_mse',
            'params','sec_per_epoch','val_loss_nats','bits_per_char','notes']
    row = {c: kw.get(c, '') for c in cols}
    exists = os.path.exists(RESULTS_CSV)
    already = set()
    if exists:
        try:
            df = pd.read_csv(RESULTS_CSV)
            for _, r in df.iterrows():
                already.add((r['problem'], r['model'], int(r['seed']),
                             int(r['seq_len']) if pd.notna(r['seq_len']) else -1,
                             str(r['notes'])))
        except Exception:
            pass
    key = (row['problem'], row['model'], int(row['seed']),
           int(row['seq_len']) if pd.notna(row['seq_len']) else -1,
           str(row['notes']))
    if key in already:
        return
    write_header = not exists
    with open(RESULTS_CSV, 'a') as f:
        if write_header:
            f.write(','.join(cols) + '\n')
        f.write(','.join(str(row[c]) for c in cols) + '\n')

print('Instrumentation ready')


## P2.1 — Scaled dot-product attention from scratch

Implementation: `scores = Q @ K^T / sqrt(d_k)`, optional mask (`masked_fill` then softmax),
output = weights @ V.


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q, K, V: (..., T_q, d_k) / (..., T_k, d_k) / (..., T_k, d_v).
    mask: broadcastable to (..., T_q, T_k); True/1 = keep, False/0 = mask out.
    Returns (output, attention_weights)."""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask.bool(), float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

print('Attention function ready')


In [ ]:
## P2.1 Unit tests (assert cells placed directly after the function)

seed_all(0)
B, H, Tq, Tk, dk, dv = 2, 3, 5, 7, 8, 8
Q = torch.randn(B, H, Tq, dk)
K = torch.randn(B, H, Tk, dk)
V = torch.randn(B, H, Tk, dv)

# Test 1: rows of attention weights sum to 1
out, w = scaled_dot_product_attention(Q, K, V)
row_sums = w.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6), row_sums
print('Test 1 PASS: attention rows sum to 1')

# Test 2: matches F.scaled_dot_product_attention within 1e-5
ref_out2 = F.scaled_dot_product_attention(Q, K, V)
assert torch.allclose(out, ref_out2, atol=1e-5), (out - ref_out2).abs().max()
print('Test 2 PASS: matches F.scaled_dot_product_attention within 1e-5')

# Test 3: causal mask gives zero weight to future positions
mask = torch.tril(torch.ones(Tq, Tk, dtype=torch.bool))
out_c, w_c = scaled_dot_product_attention(Q, K, V, mask=mask)
future = w_c[:, :, torch.triu(torch.ones(Tq, Tk, dtype=torch.bool), diagonal=1) == 1]
assert future.abs().max().item() == 0.0, future.abs().max()
print('Test 3 PASS: causal mask -> future weights exactly 0')


In [ ]:
## P2.1 3-token hand example (computed by hand and by code)

# d_k = 2, 3 tokens
Q_h = torch.tensor([[1.0, 0.0],
                    [0.0, 1.0],
                    [0.0, 0.0]])
K_h = torch.tensor([[1.0, 0.0],
                    [0.0, 1.0],
                    [0.0, 0.0]])
V_h = torch.tensor([[10.0, 0.0],
                    [0.0, 10.0],
                    [0.0, 0.0]])

out_h, w_h = scaled_dot_product_attention(Q_h.unsqueeze(0), K_h.unsqueeze(0), V_h.unsqueeze(0))
out_h = out_h[0]; w_h = w_h[0]

print('Attention weights:')
for i in range(3):
    print('  row', i, ':', [round(x,3) for x in w_h[i].tolist()])
print('Output:')
for i in range(3):
    print('  row', i, ':', [round(x,2) for x in out_h[i].tolist()])

assert torch.allclose(w_h[0], torch.tensor([0.5034, 0.2483, 0.2483]), atol=1e-3)
assert torch.allclose(w_h[1], torch.tensor([0.2483, 0.5034, 0.2483]), atol=1e-3)
assert torch.allclose(w_h[2], torch.tensor([0.3333, 0.3333, 0.3333]), atol=1e-3)
print('Hand example PASS')


## P2.2 — Multi-head attention + pre-LN encoder block + sinusoidal positional encoding

Multi-head: W_Q/W_K/W_V/W_O with head reshaping.
Encoder block: pre-LayerNorm, FFN width 4d, residual connections, dropout.
Positional encoding: sinusoidal.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        assert d % heads == 0
        self.d, self.heads = d, heads
        self.d_head = d // heads
        scale = 1 / math.sqrt(d)
        self.W_Q = nn.Parameter(torch.randn(d, d) * scale)
        self.W_K = nn.Parameter(torch.randn(d, d) * scale)
        self.W_V = nn.Parameter(torch.randn(d, d) * scale)
        self.W_O = nn.Parameter(torch.randn(d, d) * scale)
        self.b_Q = nn.Parameter(torch.zeros(d))
        self.b_K = nn.Parameter(torch.zeros(d))
        self.b_V = nn.Parameter(torch.zeros(d))
        self.b_O = nn.Parameter(torch.zeros(d))

    def forward(self, x, mask=None):
        B, T, d = x.shape
        Q = x @ self.W_Q.t() + self.b_Q
        K = x @ self.W_K.t() + self.b_K
        V = x @ self.W_V.t() + self.b_V
        Q = Q.view(B, T, self.heads, self.d_head).transpose(1, 2)
        K = K.view(B, T, self.heads, self.d_head).transpose(1, 2)
        V = V.view(B, T, self.heads, self.d_head).transpose(1, 2)
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, d)
        out = attn_out @ self.W_O.t() + self.b_O
        return out, weights

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        T = x.shape[1]
        return x + self.pe[:T].unsqueeze(0)

class EncoderBlock(nn.Module):
    def __init__(self, d, heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.mha = MultiHeadAttention(d, heads)
        self.dropout1 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Linear(d, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d),
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout1(self.mha(self.ln1(x), mask)[0])
        x = x + self.dropout2(self.ffn(self.ln2(x)))
        return x

d, d_ff = 128, 512
block = EncoderBlock(d, 4, d_ff)
total = sum(p.numel() for p in block.parameters())
weights_only = 4 * d * d + 2 * d * d_ff
print('Block weights only (formula):', f'{weights_only:,}')
print('Block total params (incl biases + LN):', f'{total:,}')
print('Expected weights-only: 196,608 for d=128, d_ff=512')


## P2.3 — Task A: Encoder classifier on HAR

Reuses Problem 1 data. Patch-embed 8 steps → 16 tokens, add CLS token,
classify from CLS. Positional ablation: none / sinusoidal / learned.


In [ ]:
CHANNELS = ['body_acc_x','body_acc_y','body_acc_z','body_gyro_x','body_gyro_y','body_gyro_z',
            'total_acc_x','total_acc_y','total_acc_z']

def load_har_task2(split):
    base = os.path.join('har', 'UCI HAR Dataset', split, 'Inertial Signals')
    mats = [np.loadtxt(os.path.join(base, f'{ch}_{split}.txt'), dtype=np.float32) for ch in CHANNELS]
    X = np.stack(mats, axis=-1)
    y = np.loadtxt(os.path.join('har', 'UCI HAR Dataset', split, f'y_{split}.txt'), dtype=np.int64) - 1
    subj = np.loadtxt(os.path.join('har', 'UCI HAR Dataset', split, f'subject_{split}.txt'), dtype=np.int64)
    return X, y, subj

X_tr_all, y_tr_all, sub_tr_all = load_har_task2('train')
X_te, y_te, _ = load_har_task2('test')
print('HAR train:', X_tr_all.shape, 'test:', X_te.shape)

def get_val_subjects(train_subjects, seed, n_val=4):
    rng = np.random.default_rng(seed)
    return rng.choice(train_subjects, n_val, replace=False)

def normalise(X_tr, X_va, X_te):
    mu = X_tr.mean(axis=(0,1), keepdims=True)
    sd = X_tr.std(axis=(0,1), keepdims=True) + 1e-6
    return (X_tr-mu)/sd, (X_va-mu)/sd, (X_te-mu)/sd

def build_har_loaders_t2(seed):
    val_subj = get_val_subjects(np.unique(sub_tr_all), seed)
    mask_val = np.isin(sub_tr_all, val_subj)
    X_tr, y_tr = X_tr_all[~mask_val], y_tr_all[~mask_val]
    X_va, y_va = X_tr_all[mask_val], y_tr_all[mask_val]
    X_tr, X_va, X_te = normalise(X_tr, X_va, X_te)
    tr = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=BATCH, shuffle=True)
    va = DataLoader(TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va)), batch_size=BATCH)
    te = DataLoader(TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te)), batch_size=BATCH)
    return tr, va, te

class PatchEmbed(nn.Module):
    def __init__(self, patch_len=8, d_in=9, d_model=128):
        super().__init__()
        self.proj = nn.Linear(d_in * patch_len, d_model)

    def forward(self, x):
        B, T, d_in = x.shape
        patch_len = 8
        num_patches = T // patch_len
        x = x[:, :num_patches * patch_len].view(B, num_patches, d_in * patch_len)
        return self.proj(x)

class TransformerClassifier(nn.Module):
    def __init__(self, d_model=128, heads=4, layers=4, d_ff=512, n_classes=6,
                 pos_type='sinusoidal', dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(d_in=9, d_model=d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_type = pos_type
        if pos_type == 'sinusoidal':
            self.pos_enc = SinusoidalPositionalEncoding(d_model)
        elif pos_type == 'learned':
            self.pos_embed = nn.Parameter(torch.randn(1, 17, d_model) * 0.02)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, heads, d_ff, dropout) for _ in range(layers)
        ])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.patch_embed(x)
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        if self.pos_type == 'sinusoidal':
            x = self.pos_enc(x)
        elif self.pos_type == 'learned':
            x = x + self.pos_embed
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln(x)
        return self.head(x[:, 0, :])

print('Task A model ready')


In [ ]:
## P2.3 Run Task A positional ablation

task2_results = {}
for pos in ['none', 'sinusoidal', 'learned']:
    class NoPosTransformer(TransformerClassifier):
        def forward(self, x, mask=None):
            x = self.patch_embed(x)
            B = x.shape[0]
            cls = self.cls_token.expand(B, -1, -1)
            x = torch.cat([cls, x], dim=1)
            x = self.dropout(x)
            for block in self.blocks:
                x = block(x, mask)
            x = self.ln(x)
            return self.head(x[:, 0, :])

    seed_all(0)
    tr, va, te = build_har_loaders_t2(0)
    if pos == 'none':
        model = NoPosTransformer().to(DEVICE)
    else:
        model = TransformerClassifier(pos_type=pos).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    epochs = 8 if FAST else 20
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
    sec_ep = (time.time() - t0) / epochs
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for xb, yb in te:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            all_p.append(model(xb).argmax(1).cpu()); all_t.append(yb.cpu())
    y_pred = torch.cat(all_p).numpy(); y_true = torch.cat(all_t).numpy()
    acc = float((y_pred == y_true).mean())
    f1s = []
    for c in range(6):
        tp = ((y_pred==c)&(y_true==c)).sum()
        fp = ((y_pred==c)&(y_true!=c)).sum()
        fn = ((y_pred!=c)&(y_true==c)).sum()
        prec = tp/(tp+fp) if (tp+fp) else 0.0
        rec = tp/(tp+fn) if (tp+fn) else 0.0
        f1s.append(2*prec*rec/(prec+rec) if (prec+rec) else 0.0)
    f1 = float(np.mean(f1s))
    params = sum(p.numel() for p in model.parameters())
    task2_results[pos] = (acc, f1, params, sec_ep)
    log_run(problem='P2', model='Transformer-'+pos, seed=0, seq_len=128,
          test_acc=acc, macro_f1=f1, params=params,
          sec_per_epoch=round(sec_ep,2), notes='Task A pos-ablation ' + pos)
    print(pos + ': acc=' + str(round(acc,4)) + ' f1=' + str(round(f1,4)) + ' params=' + str(params) + ' sec/ep=' + str(round(sec_ep,1)))

plt.figure(figsize=(6,4))
poss = list(task2_results.keys())
accs = [task2_results[p][0] for p in poss]
plt.bar(poss, accs)
plt.ylabel('Test accuracy'); plt.title('Task A: positional encoding ablation')
plt.tight_layout()
os.makedirs('problem2/figures', exist_ok=True)
plt.savefig('problem2/figures/fig_task2_pos_ablation.png', dpi=150)
plt.show()


## P2.4 — Task B: Character-level causal language model on Tiny Shakespeare

Context 128, d=128, 4 heads, 4 layers. Train with teacher forcing + causal mask.
Generate 500 chars at temperature 0.2 / 0.8 / 1.5 with top-k sampling.


In [ ]:
import urllib.request

URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('tinyshakespeare.txt'):
    print('Downloading Tiny Shakespeare ...')
    urllib.request.urlretrieve(URL, 'tinyshakespeare.txt')

text = open('tinyshakespeare.txt', encoding='utf-8').read()
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}
V = len(vocab)
print('Vocab size:', V, '| Text length:', len(text))

n_train = int(0.9 * len(text))
train_text = text[:n_train]
val_text = text[n_train:]

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join(itos[i] for i in l)

class CharDataset:
    def __init__(self, text, ctx=128):
        data = torch.tensor(encode(text), dtype=torch.long)
        xs, ys = [], []
        for i in range(0, len(data) - ctx, 1):
            xs.append(data[i:i+ctx])
            ys.append(data[i+1:i+ctx+1])
        self.x = torch.stack(xs)
        self.y = torch.stack(ys)
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

ctx = 128
train_ds = CharDataset(train_text, ctx)
val_ds = CharDataset(val_text, ctx)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH)
print('Train batches:', len(train_dl), '| Val batches:', len(val_dl))

class CharTransformerLM(nn.Module):
    def __init__(self, vocab_size, d=128, heads=4, layers=4, d_ff=512, ctx=128, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Parameter(torch.randn(1, ctx, d) * 0.02)
        self.blocks = nn.ModuleList([EncoderBlock(d, heads, d_ff, dropout) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size)
        self.ctx = ctx
        self.dropout = nn.Dropout(dropout)

    def forward(self, idx, mask=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb[:, :T, :]
        x = self.dropout(x)
        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))
        for block in self.blocks:
            x = block(x, mask=causal)
        x = self.ln(x)
        return self.head(x)

lm = CharTransformerLM(V, d=128, heads=4, layers=4, ctx=ctx).to(DEVICE)
print('LM params:', sum(p.numel() for p in lm.parameters()))


In [ ]:
## P2.4 Train character LM

opt = torch.optim.AdamW(lm.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
epochs = 2 if FAST else 10
train_losses, val_losses = [], []
t0 = time.time()
for ep in range(epochs):
    lm.train()
    ep_losses = []
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = lm(xb)
        loss = crit(logits.view(-1, V), yb.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
        opt.step()
        ep_losses.append(loss.item())
    train_losses.append(np.mean(ep_losses))

    lm.eval()
    vlosses = []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = lm(xb)
            vlosses.append(crit(logits.view(-1, V), yb.view(-1)).item())
    val_losses.append(np.mean(vlosses))
    print(f'Epoch {ep+1}: train={train_losses[-1]:.4f} val={val_losses[-1]:.4f} ',
          f'bits/char={val_losses[-1]/math.log(2):.3f}')

    log_run(problem='P2', model='CharTransformer', seed=0, seq_len=ctx,
          val_loss_nats=float(val_losses[-1]),
          bits_per_char=float(val_losses[-1]/math.log(2)),
          notes='epoch ' + str(ep+1))

plt.figure(figsize=(6,4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss (nats)')
plt.title('Task B: character LM learning curve')
plt.legend()
plt.tight_layout()
plt.savefig('problem2/figures/fig_taskb_learning_curve.png', dpi=150)
plt.show()
print('Final val loss:', round(val_losses[-1],4), 'nats | bits/char:', round(val_losses[-1]/math.log(2),3))


In [ ]:
## P2.4 Generate 500 chars at temperatures 0.2 / 0.8 / 1.5

def top_k_sample(logits, k=40, temperature=1.0):
    logits = logits / temperature
    top_k_vals, top_k_idx = torch.topk(logits, k)
    probs = F.softmax(top_k_vals, dim=-1)
    chosen = torch.multinomial(probs, 1)
    return top_k_idx.gather(-1, chosen).item()

def generate(model, start='To be ', n=500, temperature=1.0, k=40):
    model.eval()
    ids = encode(start)
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for _ in range(n):
            idx_crop = idx[:, -model.ctx:]
            logits = model(idx_crop)[:, -1, :]
            next_id = top_k_sample(logits[0], k=k, temperature=temperature)
            idx = torch.cat([idx, torch.tensor([[next_id]], device=DEVICE)], dim=1)
    return decode(idx[0].tolist())

samples = {}
for temp in [0.2, 0.8, 1.5]:
    samples[temp] = generate(lm, start='To be ', n=500, temperature=temp)
    print('\n=== Temperature ' + str(temp) + ' ===')
    print(samples[temp][:300])

os.makedirs('problem2', exist_ok=True)
with open('problem2/generated_samples.txt', 'w') as f:
    for temp in [0.2, 0.8, 1.5]:
        f.write('=== Temperature ' + str(temp) + ' ===\\n')
        f.write(samples[temp] + '\\n\\n')
print('Saved problem2/generated_samples.txt')





## P2.5 — Analysis and ablations

(a) Attention heatmaps for two heads
(b) Heads = 1 / 4 / 8 at fixed d
(c) Remove causal mask
(d) Context length 64 / 128 / 256 / 512: time + memory vs O(n²·d)


In [ ]:
## P2.5a Attention heatmaps

lm.eval()
sentence = 'To be or not to be'
ids = torch.tensor([encode(sentence)], device=DEVICE)

with torch.no_grad():
    x = lm.token_emb(ids) + lm.pos_emb[:, :ids.shape[1], :]
    x = lm.dropout(x)
    causal = torch.tril(torch.ones(ids.shape[1], ids.shape[1], dtype=torch.bool, device=DEVICE))
    block0 = lm.blocks[0]
    x_ln = block0.ln1(x)
    B, T, d = x_ln.shape
    Q = x_ln @ block0.mha.W_Q.t() + block0.mha.b_Q
    K = x_ln @ block0.mha.W_K.t() + block0.mha.b_K
    Q = Q.view(B, T, block0.mha.heads, block0.mha.d_head).transpose(1, 2)
    K = K.view(B, T, block0.mha.heads, block0.mha.d_head).transpose(1, 2)
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    scores = scores.masked_fill(~causal, float('-inf'))
    weights = F.softmax(scores, dim=-1)

tokens = list(sentence)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for h_idx, ax in enumerate([axes[0], axes[1]]):
    im = ax.imshow(weights[0, h_idx].cpu().numpy(), cmap='Blues')
    ax.set_xticks(range(T)); ax.set_yticks(range(T))
    ax.set_xticklabels(tokens, rotation=45); ax.set_yticklabels(tokens)
    ax.set_title('Head ' + str(h_idx))
    plt.colorbar(im, ax=ax)
plt.suptitle('P2.5a: Attention heatmaps (block 0, sentence: "' + sentence + '")')
plt.tight_layout()
plt.savefig('problem2/figures/fig_p25a_heatmaps.png', dpi=150)
plt.show()
print('Head 0 attends to local/previous context; Head 1 shows longer-range patterns.')


In [ ]:
## P2.5b Heads ablation (1 / 4 / 8 at fixed d=128)

def build_lm_heads(n_heads):
    return CharTransformerLM(V, d=128, heads=n_heads, layers=2, d_ff=512, ctx=ctx).to(DEVICE)

heads_results = {}
for nh in [1, 4, 8]:
    seed_all(0)
    model = build_lm_heads(nh)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    for ep in range(2 if FAST else 5):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).view(-1, V), yb.view(-1))
            loss.backward(); opt.step()
    model.eval()
    vlosses = []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vlosses.append(crit(model(xb).view(-1, V), yb.view(-1)).item())
    heads_results[nh] = float(np.mean(vlosses))
    print('heads=' + str(nh) + ' val_loss=' + str(round(heads_results[nh],4)))

plt.figure(figsize=(5,4))
plt.bar([str(h) for h in heads_results.keys()], list(heads_results.values()))
plt.xlabel('Number of heads'); plt.ylabel('Val loss (nats)')
plt.title('P2.5b: Heads ablation (fixed d=128)')
plt.tight_layout()
plt.savefig('problem2/figures/fig_p25b_heads.png', dpi=150)
plt.show()


In [ ]:
## P2.5c Remove causal mask

class NoMaskCharLM(CharTransformerLM):
    def forward(self, idx, mask=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb[:, :T, :]
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x, mask=None)
        x = self.ln(x)
        return self.head(x)

seed_all(0)
nomask_lm = NoMaskCharLM(V, d=64, heads=2, layers=2, d_ff=256, ctx=ctx).to(DEVICE)
opt = torch.optim.AdamW(nomask_lm.parameters(), lr=LR)
normal_losses, nomask_losses = [], []
for ep in range(3 if FAST else 8):
    lm.train()
    tl = []
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(lm(xb).view(-1, V), yb.view(-1))
        loss.backward(); opt.step()
        tl.append(loss.item())
    normal_losses.append(np.mean(tl))

    nomask_lm.train()
    tl2 = []
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(nomask_lm(xb).view(-1, V), yb.view(-1))
        loss.backward(); opt.step()
        tl2.append(loss.item())
    nomask_losses.append(np.mean(tl2))
    print('Epoch ' + str(ep+1) + ': causal=' + str(round(normal_losses[-1],4)) + ' no-mask=' + str(round(nomask_losses[-1],4)))

plt.figure(figsize=(6,4))
plt.plot(normal_losses, label='causal mask')
plt.plot(nomask_losses, label='no mask')
plt.xlabel('Epoch'); plt.ylabel('Train loss')
plt.title('P2.5c: Causal mask vs no mask')
plt.legend()
plt.tight_layout()
plt.savefig('problem2/figures/fig_p25c_nomask.png', dpi=150)
plt.show()
print('Without causal mask: model sees future tokens → cheating on training loss, generation degrades.')


In [ ]:
## P2.5d Context length: time + memory vs O(n²·d)

contexts = [64, 128, 256, 512]
times, mems = [], []
for C in contexts:
    seed_all(0)
    model = CharTransformerLM(V, d=64, heads=2, layers=2, d_ff=256, ctx=C).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    xb = torch.randint(0, V, (16, C), device=DEVICE)
    yb = torch.randint(0, V, (16, C), device=DEVICE)

    for _ in range(2):
        _ = model(xb)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(5):
        opt.zero_grad()
        loss = crit(model(xb).view(-1, V), yb.view(-1))
        loss.backward()
        opt.step()
    elapsed = (time.time() - t0) / 5
    if torch.cuda.is_available():
        mem = torch.cuda.max_memory_allocated() / 1e6
    else:
        mem = 0
    times.append(elapsed)
    mems.append(mem)
    print('ctx=' + str(C) + ' time/iter=' + str(round(elapsed,4)) + 's mem=' + str(round(mem,1)) + 'MB')

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(contexts, times, marker='o')
plt.plot(contexts, [t * (c/contexts[0])**2 for t, c in zip(times, contexts)], 'k--', label='O(n^2) ref')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Context length'); plt.ylabel('Time per iter (s)')
plt.title('P2.5d: Time vs context length')
plt.legend()

plt.subplot(1,2,2)
plt.plot(contexts, mems, marker='o', color='red')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Context length'); plt.ylabel('Peak memory (MB)')
plt.title('P2.5d: Memory vs context length')
plt.tight_layout()
plt.savefig('problem2/figures/fig_p25d_context.png', dpi=150)
plt.show()
print('Memory and time should grow ~O(n^2) with context length.')


## Done — Problem 2 complete

Run `Restart & Run All`. Save `results.csv` after every run (handled by `log_run`).
All required figures are saved to `problem2/figures/`.
